# Cardiac Function (EF) Prediction with Missing Frames
### Baseline (R3D-18) vs. Mamba-Inspired Selective SSM (GapSSM)

Configured for your Kaggle dataset structure:
- **Project Input**: `/kaggle/input/ai_projectv1/AI_project`
- **Dataset Input**: `/kaggle/input/echonet/EchoNet-Dynamic`
- **Writable Output**: `/kaggle/working/cardiac_runs`

## 1. Environment & GPU Setup

In [ ]:
import os
import sys
import time
from pathlib import Path
import torch

print(f"[CHECKPOINT] Python Version: {sys.version.split()[0]}")
print(f"[CHECKPOINT] PyTorch Version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"[CHECKPOINT] CUDA Available: {cuda_available}")
if cuda_available:
    print(f"  - GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"  - Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("  - Running in CPU mode. (Note: Enable GPU Accelerator under Kaggle Session options for faster training)")

## 2. Locate Project & Set Up Python Path

Detects `ai_projectv1/AI_project` in `/kaggle/input` and adds `src` and `scripts` to `sys.path`.

In [ ]:
CWD = Path.cwd()

# Search candidate project directories matching the Kaggle dataset structure
possible_roots = [
    Path('/kaggle/input/ai_projectv1/AI_project'),
    Path('/kaggle/input/ai-projectv1/AI_project'),
    Path('/kaggle/input/ai_projectv1'),
    Path('/kaggle/input/ai-projectv1'),
    *Path('/kaggle/input').glob('**/AI_project'),
    *Path('/kaggle/input').glob('*'),
    Path('/kaggle/working'),
    CWD,
    CWD.parent
]

PROJECT_ROOT = None
for p in possible_roots:
    if p.exists() and (p / 'src' / 'cardiac').exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate AI_project with 'src/cardiac'. "
        "Check that the 'ai_projectv1' dataset is attached under Input!"
    )

print(f"[CHECKPOINT] Located Project Root at: {PROJECT_ROOT}")

# Add src and scripts to sys.path
SRC_PATH = str(PROJECT_ROOT / 'src')
SCRIPTS_PATH = str(PROJECT_ROOT / 'scripts')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
if SCRIPTS_PATH not in sys.path:
    sys.path.insert(0, SCRIPTS_PATH)

# Set output directory in Kaggle writable workspace
WORKSPACE = Path('/kaggle/working') if Path('/kaggle/working').exists() else CWD
RUNS = WORKSPACE / 'cardiac_runs'
RUNS.mkdir(parents=True, exist_ok=True)
print(f"[CHECKPOINT] Writable Output Directory: {RUNS}")

## 3. Dataset Discovery (`EchoNet/EchoNet-Dynamic`)

Detects `EchoNet/EchoNet-Dynamic` from `/kaggle/input/`. If not found, falls back to generating a demo synthetic fixture.

In [ ]:
from make_fixture import create

# Exact search paths based on Kaggle input: EchoNet -> EchoNet-Dynamic
candidate_data_dirs = [
    Path('/kaggle/input/echonet/EchoNet-Dynamic'),
    Path('/kaggle/input/EchoNet/EchoNet-Dynamic'),
    Path('/kaggle/input/echonet/echonet-dynamic'),
    Path('/kaggle/input/echonet-dynamic/EchoNet-Dynamic'),
    Path('/kaggle/input/echonet-dynamic'),
    Path('/kaggle/input/echonet'),
    Path('/kaggle/input/EchoNet'),
    *Path('/kaggle/input').glob('**/EchoNet-Dynamic'),
    *Path('/kaggle/input').glob('**/*EchoNet*'),
    PROJECT_ROOT / 'data' / 'EchoNet-Dynamic',
    CWD / 'data' / 'EchoNet-Dynamic'
]

DATA_DIR = None
IS_SYNTHETIC = False

for cand in candidate_data_dirs:
    if cand.is_dir() and (cand / 'FileList.csv').exists():
        DATA_DIR = cand
        print(f"[CHECKPOINT] Found real EchoNet-Dynamic dataset at: {DATA_DIR}")
        break

if DATA_DIR is None:
    print("[INFO] Real EchoNet-Dynamic dataset not found in inputs.")
    print("[CHECKPOINT] Generating synthetic demo fixture for end-to-end verification...")
    DATA_DIR = create(RUNS / 'demo_dataset')
    IS_SYNTHETIC = True
    print(f"[CHECKPOINT] Demo synthetic fixture created at: {DATA_DIR}")

print(f"[STATUS] Ready with {'SYNTHETIC DEMO' if IS_SYNTHETIC else 'REAL ECHONET'} dataset at: {DATA_DIR}")

## 4. Build Dataset Manifest & Freeze Test Mask Bank

In [ ]:
from cardiac.data.manifest import build_manifest
from cardiac.experiments.bank import build_bank

MANIFEST_PATH = RUNS / 'manifest.json'
TEST_BANK_PATH = RUNS / 'test_bank.json'

# 1. Build manifest (audits video files, checks frames, extracts ED/ES traces)
print("[CHECKPOINT] Building Dataset Manifest...")
manifest_data = build_manifest(
    root=DATA_DIR,
    output=MANIFEST_PATH,
    allow_subset=True,
    synthetic=IS_SYNTHETIC
)
print(f"[CHECKPOINT] Manifest built: {len(manifest_data['records'])} valid videos (Splits: {manifest_data['split_counts']})")

# 2. Build test bank (freezes identical missing-frame masks for fair comparison)
print("[CHECKPOINT] Freezing Mask Bank for Evaluation...")
clip_len = 8 if IS_SYNTHETIC else 32
bank_data = build_bank(
    manifest_path=MANIFEST_PATH,
    output=TEST_BANK_PATH,
    length=clip_len,
    stride=2,
    seed=2026,
    split='TEST'
)
print(f"[CHECKPOINT] Bank created with {len(bank_data['entries'])} test video entries.")

## 5. Model 1: Train 3D CNN Baseline (R3D-18)

Trains the baseline model. Epoch checkpoint files (`checkpoint_epoch_*.pt`), `last.pt`, and `best.pt` are saved in `/kaggle/working/cardiac_runs/baseline/`.

In [ ]:
from cardiac.training.engine import train
from cardiac.utils.io import write_json

BASELINE_DIR = RUNS / 'baseline'
BASELINE_CONFIG_PATH = RUNS / 'baseline_config.json'

# Configure Baseline
baseline_config = {
    'model': 'baseline', # R3D-18 video backbone
    'corruption': False,
    'length': clip_len,
    'stride': 2,
    'size': 32 if IS_SYNTHETIC else 112,
    'epochs': 3 if IS_SYNTHETIC else 25,
    'learning_rate': 1e-4,
    'batch_size': 2 if IS_SYNTHETIC else (8 if cuda_available else 2),
    'accumulation': 2,
    'patience': 5,
    'save_epochs': True
}
write_json(BASELINE_CONFIG_PATH, baseline_config)

print("[CHECKPOINT] Starting Baseline (R3D-18) Training...", flush=True)
sys.stdout.flush()
baseline_meta = train(
    root=DATA_DIR,
    manifest_path=MANIFEST_PATH,
    config_path=BASELINE_CONFIG_PATH,
    output=BASELINE_DIR,
    device='auto',
    synthetic=IS_SYNTHETIC,
    limit=8 if IS_SYNTHETIC else None,
    save_epochs=True
)
print(f"[CHECKPOINT] Baseline training complete! Best Val MAE: {baseline_meta.get('best_val_mae', 'N/A'):.4f}", flush=True)


## 6. Model 2: Train Mamba-Inspired Model (GapSSM)

Trains the Gap-aware Selective State-Space Model. Epoch checkpoint files are saved in `/kaggle/working/cardiac_runs/mamba/`.

In [ ]:
MAMBA_DIR = RUNS / 'mamba'
MAMBA_CONFIG_PATH = RUNS / 'mamba_config.json'

# Configure Mamba (GapSSM)
mamba_config = {
    'model': 'mamba',
    'width': 16 if IS_SYNTHETIC else 128,
    'state_size': 4 if IS_SYNTHETIC else 16,
    'layers': 1 if IS_SYNTHETIC else 2,
    'encoder': 'tiny' if IS_SYNTHETIC else 'resnet18',
    'use_mask': True,
    'use_gap': True,
    'hold_missing': True,
    'corruption': True,
    'length': clip_len,
    'stride': 2,
    'size': 32 if IS_SYNTHETIC else 112,
    'epochs': 3 if IS_SYNTHETIC else 25,
    'learning_rate': 1e-4,
    'batch_size': 2 if IS_SYNTHETIC else (8 if cuda_available else 2),
    'accumulation': 2,
    'patience': 5,
    'save_epochs': True
}
write_json(MAMBA_CONFIG_PATH, mamba_config)

print("[CHECKPOINT] Starting Mamba (GapSSM) Training...", flush=True)
sys.stdout.flush()
mamba_meta = train(
    root=DATA_DIR,
    manifest_path=MANIFEST_PATH,
    config_path=MAMBA_CONFIG_PATH,
    output=MAMBA_DIR,
    device='auto',
    synthetic=IS_SYNTHETIC,
    limit=8 if IS_SYNTHETIC else None,
    save_epochs=True
)
print(f"[CHECKPOINT] Mamba training complete! Best Val MAE: {mamba_meta.get('best_val_mae', 'N/A'):.4f}", flush=True)


## 7. Evaluate Both Models Under Missing Frames

Tests robustness across 4 imputation strategies (`zero`, `removal`, `repeat`, `interpolate`) and missing frame ratios (0%, 10%, 25%, 50%, 75%).

In [ ]:
from cardiac.evaluation.runner import evaluate

handlings = ['zero', 'repeat'] if IS_SYNTHETIC else ['zero', 'removal', 'repeat', 'interpolate']
mc_passes = 3 if IS_SYNTHETIC else 10

eval_runs = [
    ('baseline', BASELINE_DIR / 'best.pt', BASELINE_DIR / 'evaluation'),
    ('mamba', MAMBA_DIR / 'best.pt', MAMBA_DIR / 'evaluation')
]

prediction_csvs = []
for model_name, ckpt_path, eval_dir in eval_runs:
    print(f"\n[CHECKPOINT] Evaluating {model_name} on Test Bank...")
    eval_meta = evaluate(
        root=DATA_DIR,
        manifest_path=MANIFEST_PATH,
        bank_path=TEST_BANK_PATH,
        checkpoint_path=ckpt_path,
        output=eval_dir,
        device='auto',
        handlings=handlings,
        mc_passes=mc_passes,
        synthetic=IS_SYNTHETIC,
        model_name=model_name,
        allow_hash_mismatch=True
    )
    prediction_csvs.append(eval_dir / 'predictions.csv')
    print(f"[CHECKPOINT] {model_name} evaluation complete -> {eval_dir / 'predictions.csv'}")

## 8. Generate Reports & Comparative Visualizations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from cardiac.evaluation.reporting import report

REPORT_DIR = RUNS / 'comparison'
report_summary = report(prediction_csvs, REPORT_DIR)

# Plot Training & Validation Curves
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for run_dir, label, color in [(BASELINE_DIR, 'Baseline (R3D-18)', 'blue'), (MAMBA_DIR, 'Mamba (GapSSM)', 'darkorange')]:
    csv_file = run_dir / 'training.csv'
    if csv_file.exists():
        hist = pd.read_csv(csv_file)
        ax[0].plot(hist['epoch'], hist['train_mse_fraction'], label=f'{label} Train MSE', color=color, linestyle='--')
        ax[1].plot(hist['epoch'], hist['val_mae_pp'], label=f'{label} Val MAE', color=color, linewidth=2)

ax[0].set_title('Training MSE Loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('MSE Loss')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].set_title('Validation MAE (Percentage Points)')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('MAE (pp)')
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.tight_layout()
curve_path = REPORT_DIR / 'training_comparison.png'
plt.savefig(curve_path, dpi=150)
plt.show()
print(f"[CHECKPOINT] Comparison curves saved to: {curve_path}")

# Display Metrics Table
metrics_csv = REPORT_DIR / 'tables' / 'metrics.csv'
if metrics_csv.exists():
    df_metrics = pd.read_csv(metrics_csv)
    print("\n--- Comparative Performance Summary (Sample) ---")
    display_cols = [c for c in ['model', 'handling', 'ratio', 'mae', 'rmse', 'r2', 'pearson'] if c in df_metrics.columns]
    print(df_metrics[display_cols].head(15).to_string(index=False))
    df_metrics.to_csv(REPORT_DIR / 'final_metrics_summary.csv', index=False)

## 9. Package Artifacts & Checkpoints for 1-Click Download

In [ ]:
import shutil

# Create zip archive of all outputs in /kaggle/working for download
OUTPUT_ZIP = WORKSPACE / 'cardiac_experiment_results'
shutil.make_archive(str(OUTPUT_ZIP), 'zip', str(RUNS))
print(f"[CHECKPOINT] All checkpoints and evaluation outputs packaged at:")
print(f"  -> {OUTPUT_ZIP}.zip")
print("[SUCCESS] Pipeline execution finished successfully! Download the zip from the Kaggle Output tab.")